# SMS Spam Detection Model Training & Evaluation
This notebook cleans the dataset, extracts TF-IDF features, trains Multinomial Naive Bayes and Logistic Regression models, evaluates their metrics, and saves the best model.

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


## 1. Load Dataset

In [ ]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'text']
df['target'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()


## 2. Text Preprocessing

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
df.head()


## 3. Train Test Split & TF-IDF Feature Extraction

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_text'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


## 4. Model Training & Comparison

In [ ]:
# 1. Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

# 2. Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print('--- Naive Bayes Classification Report ---')
print(classification_report(y_test, y_pred_nb, target_names=['Ham', 'Spam']))
print('--- Logistic Regression Classification Report ---')
print(classification_report(y_test, y_pred_lr, target_names=['Ham', 'Spam']))


## 5. Confusion Matrix Visualization

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(y_test, y_pred_nb), annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('Naive Bayes Confusion Matrix')
plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Greens', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('Logistic Regression Confusion Matrix')
plt.tight_layout()
plt.show()
